In [261]:
import pandas as pd
import numpy as np
import arcticdb as adb
import nasdaqdatalink as ndl
import matplotlib.pyplot as plt
from jedi.inference.utils import to_list
%matplotlib inline
import time
from tqdm import tqdm

In [262]:
ndl.ApiConfig.api_key = "9yUyP5fu4aWYHRXYAqxx"

In [263]:
START = "2000-01-01"
END = "2026-02-15"

ARCTIC_URI = "lmdb://./database/arctic"

In [268]:
STOCK_LIBRARY = "stocks_lib"
ETF_LIBRARY = "etfs_lib"
REMOVED_LIB = "removed_stocks"
ac = adb.Arctic(ARCTIC_URI)

20260515 20:23:53.493082 33581 W arcticdb | LMDB path at /home/sysadmin/PycharmProjects/PM/database/arctic/ has already been opened in this process which is not supported by LMDB. You should only open a single Arctic instance over a given LMDB path. To continue safely, you should delete this Arctic instance and any others over the LMDB path in this process and then try again. Current process ID=[33581]


In [270]:

for lib in [STOCK_LIBRARY, ETF_LIBRARY, REMOVED_LIB]:
    if lib not in ac.list_libraries():
        ac.create_library(lib)

etf_lib = ac["etfs_lib"]
stocks_lib = ac["stocks_lib"]
removed_lib = ac["removed_stocks"]

In [246]:
stocks = ndl.get_table("SHARADAR/TICKERS", paginate=True)


In [222]:
stocks

,table,permaticker,ticker,name,exchange,isdelisted,category,cusips,siccode,sicsector,...,currency,location,lastupdated,firstadded,firstpricedate,lastpricedate,firstquarter,lastquarter,secfilings,companysite
None,,,,,,,,,,,,,,,,,,,,,
0,SFP,6399079,AMAU,LEVERAGE SHARES 2X LONG AMAT DAILY ETF,BATS,N,ETF,None,NaN,None,...,USD,Connecticut; U.S.A,2026-05-15,2026-05-13,2026-05-12,2026-05-15,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
1,SFP,6399078,ATC,GRANITESHARES AUTOCALLABLE COIN ETF,NASDAQ,N,ETF,None,NaN,None,...,USD,New York; U.S.A,2026-05-15,2026-05-13,2026-05-12,2026-05-15,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
2,SFP,6399077,CIEG,LEVERAGE SHARES 2X LONG CIEN DAILY ETF,BATS,N,ETF,None,NaN,None,...,USD,Connecticut; U.S.A,2026-05-15,2026-05-13,2026-05-12,2026-05-15,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
3,SFP,6399076,ETNG,LEVERAGE SHARES 2X LONG ETN DAILY ETF,BATS,N,ETF,None,NaN,None,...,USD,Connecticut; U.S.A,2026-05-15,2026-05-13,2026-05-12,2026-05-15,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
4,SFP,6399075,STXU,LEVERAGE SHARES 2X LONG STX DAILY ETF,BATS,N,ETF,None,NaN,None,...,USD,Connecticut; U.S.A,2026-05-15,2026-05-13,2026-05-12,2026-05-15,None,None,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61738,SEP,101501,BBUCQ,BIG BUCK BREWERY & STEAKHOUSE INC,NASDAQ,Y,Domestic Common Stock,089072102 594194201 089072201 089072300,5810.0,Retail Trade,...,USD,Michigan; U.S.A,2021-08-28,2020-06-10,1996-06-13,2021-08-27,1996-12-31,2003-09-30,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
61739,SEP,101484,NWGN1,NEWGEN RESULTS CORP,NASDAQ,Y,Domestic Common Stock,651359101,7389.0,Services,...,USD,California; U.S.A,2020-08-03,2020-08-03,1999-05-21,2000-12-20,1997-12-31,2000-09-30,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None
61740,SEP,101470,CYRS1,CYBERSENTRY INC,NYSEMKT,Y,Domestic Common Stock,23252H109,4899.0,Transportation Communications Electric Gas And...,...,USD,Florida; U.S.A,2024-04-01,2024-04-01,2000-01-26,2000-09-07,1997-12-31,2000-06-30,https://www.sec.gov/cgi-bin/browse-edgar?actio...,None


In [247]:
print(stocks["exchange"].unique())
exchange_list = stocks["exchange"].unique()
print(exchange_list)
cols = ["table", "permaticker", "ticker", "isdelisted", "sicsector" , "firstpricedate" , "lastpricedate"]
all_tickers = stocks.loc[stocks["table"] == "SEP", cols]
etfs = stocks.loc[stocks["table"] == "SFP", cols]


['BATS' 'NASDAQ' 'NYSEARCA' 'NYSE' 'NYSEMKT' 'INDEX' None 'OTC']
['BATS' 'NASDAQ' 'NYSEARCA' 'NYSE' 'NYSEMKT' 'INDEX' None 'OTC']


Defensive just in case in the histor an etf and the stock is having the same name so that we won't get the data mixed for that case
it is rare but might happen since it is a long history

In [248]:
stock_to_perma = dict(zip(
    all_tickers["ticker"],
    all_tickers["permaticker"].astype("int64"),
))
etf_to_perma = dict(zip(
    etfs["ticker"],
    etfs["permaticker"].astype("int64"),
))

In [249]:

dupes = all_tickers[all_tickers["ticker"].duplicated(keep=False)].sort_values("ticker")
print(f"{all_tickers['ticker'].duplicated().sum()} duplicate ticker rows")
dupes.head(20)


0 duplicate ticker rows


,table,permaticker,ticker,isdelisted,sicsector,firstpricedate,lastpricedate
None,,,,,,,


In [250]:
CHUNK_SIZE = 10
RETRY_ATTEMPTS = 3
DELAY_BETWEEN_CHUNKS = 0.5

In [251]:
sp500 = ndl.get_table("SHARADAR/SP500", paginate=True)
print(sp500.head(20))

           date   action ticker                              name  \
None                                                                
0    2026-05-15  current    ZTS                        ZOETIS INC   
1    2026-05-15  current   ZBRA           ZEBRA TECHNOLOGIES CORP   
2    2026-05-15  current    ZBH        ZIMMER BIOMET HOLDINGS INC   
3    2026-05-15  current    YUM                    YUM BRANDS INC   
4    2026-05-15  current    XYZ                         BLOCK INC   
5    2026-05-15  current    XYL                         XYLEM INC   
6    2026-05-15  current    XOM                  EXXON MOBIL CORP   
7    2026-05-15  current    XEL                   XCEL ENERGY INC   
8    2026-05-15  current   WYNN                  WYNN RESORTS LTD   
9    2026-05-15  current     WY                   WEYERHAEUSER CO   
10   2026-05-15  current    WTW          WILLIS TOWERS WATSON PLC   
11   2026-05-15  current    WST  WEST PHARMACEUTICAL SERVICES INC   
12   2026-05-15  current    WSM   

In [252]:
sp500["action"].value_counts()

action
historical    56690
added          1231
removed         734
current         503
Name: count, dtype: int64

In [253]:
ever_in_sp500 = sorted(
      sp500.loc[sp500["action"].isin(["current", "added", "removed"]), "ticker"].unique()
  )
len(ever_in_sp500)

1198

In [254]:
current_members = sp500.loc[sp500["action"] == "current", "ticker"].to_list()
print(current_members)

['ZTS', 'ZBRA', 'ZBH', 'YUM', 'XYZ', 'XYL', 'XOM', 'XEL', 'WYNN', 'WY', 'WTW', 'WST', 'WSM', 'WRB', 'WMT', 'WMB', 'WM', 'WFC', 'WELL', 'WEC', 'WDC', 'WDAY', 'WBD', 'WAT', 'WAB', 'VZ', 'VTRS', 'VTR', 'VST', 'VRTX', 'VRT', 'VRSN', 'VRSK', 'VMC', 'VLTO', 'VLO', 'VICI', 'VEEV', 'V', 'USB', 'URI', 'UPS', 'UNP', 'UNH', 'ULTA', 'UHS', 'UDR', 'UBER', 'UAL', 'TYL', 'TXT', 'TXN', 'TTWO', 'TTD', 'TT', 'TSN', 'TSLA', 'TSCO', 'TRV', 'TROW', 'TRMB', 'TRGP', 'TPR', 'TPL', 'TMUS', 'TMO', 'TKO', 'TJX', 'TGT', 'TFC', 'TER', 'TEL', 'TECH', 'TDY', 'TDG', 'TAP', 'T', 'SYY', 'SYK', 'SYF', 'SWKS', 'SWK', 'SW', 'STZ', 'STX', 'STT', 'STLD', 'STE', 'SRE', 'SPGI', 'SPG', 'SOLV', 'SO', 'SNPS', 'SNDK', 'SNA', 'SMCI', 'SLB', 'SJM', 'SHW', 'SCHW', 'SBUX', 'SBAC', 'SATS', 'RVTY', 'RTX', 'RSG', 'ROST', 'ROP', 'ROL', 'ROK', 'RMD', 'RL', 'RJF', 'RF', 'REGN', 'REG', 'RCL', 'QCOM', 'Q', 'PYPL', 'PWR', 'PTC', 'PSX', 'PSKY', 'PSA', 'PRU', 'PPL', 'PPG', 'POOL', 'PODD', 'PNW', 'PNR', 'PNC', 'PM', 'PLTR', 'PLD', 'PKG', 'PHM', 

In [255]:
removed = sp500.loc[
      (sp500["action"] == "removed") &
      (pd.to_datetime(sp500["date"]) >= "2000-01-01"),
      "ticker"
  ].unique()
print(len(removed))

597


In [256]:
def fetch_chunked(tickers, table, lib, mapping):
    """Download ticker history in chunks to avoid oversized API requests."""
    chunks = [tickers[i:i + CHUNK_SIZE] for i in range(0, len(tickers), CHUNK_SIZE)]
    existing = set(lib.list_symbols())
    failed_chunks = []

    print(f"Fetching {len(tickers)} tickers from {table}. {len(existing)} already in lib so we are skipping")

    for chunk in tqdm(chunks):
        todo = [t for t in chunk if t not in existing]

        if not todo:
            continue
        df = None
        for attempt in range(RETRY_ATTEMPTS):
            try:
                df = ndl.get_table(
                    table,
                    ticker=",".join(todo),
                    date={"gte": START, "lte": END},
                    paginate=True,
                )

                break
            except Exception as exc:
                if attempt ==RETRY_ATTEMPTS-1:
                    failed_chunks.append((todo, str(exc)))
                else:
                    time.sleep(2 ** attempt)

        if df is None or df.empty:
            continue

        df["date"] = pd.to_datetime(df["date"])

        for tkr, sub in df.groupby("ticker", sort=False):
            sub = (sub.drop(columns="ticker").set_index("date").sort_index())
            lib.write(tkr, sub, metadata={"permaticker": mapping.get(tkr)})
            existing.add(tkr)

        del df
        time.sleep(DELAY_BETWEEN_CHUNKS)
    print(f"done , {sum( len(c) for c, _ in failed_chunks)} failed")
    return failed_chunks

In [257]:
stock_prices = fetch_chunked(current_members, "SHARADAR/SEP", stocks_lib, stock_to_perma)

Fetching 503 tickers from SHARADAR/SEP. 0 already in lib so we are skipping


100%|██████████| 51/51 [06:11<00:00,  7.29s/it]

done , 0 failed


In [197]:
etf_prices = fetch_chunked(etfs, "SHARADAR/SFP", etf_lib, etf_to_perma)

Fetching 9300 tickers from SHARADAR/SFP. 0 already in lib so we are skipping


100%|██████████| 930/930 [11:24<00:00,  1.36it/s]    

done , 0 failed


In [271]:
removed_stocks = fetch_chunked(removed, "SHARADAR/SEP", removed_lib, stock_to_perma)

Fetching 597 tickers from SHARADAR/SEP. 0 already in lib so we are skipping


100%|██████████| 60/60 [04:59<00:00,  4.98s/it]

done , 0 failed


In [199]:
stocks.columns.to_list()

['table',
 'permaticker',
 'ticker',
 'name',
 'exchange',
 'isdelisted',
 'category',
 'cusips',
 'siccode',
 'sicsector',
 'sicindustry',
 'famasector',
 'famaindustry',
 'sector',
 'industry',
 'scalemarketcap',
 'scalerevenue',
 'relatedtickers',
 'currency',
 'location',
 'lastupdated',
 'firstadded',
 'firstpricedate',
 'lastpricedate',
 'firstquarter',
 'lastquarter',
 'secfilings',
 'companysite']

In [272]:
def add_metadata(lib, source_df, fields=None, overwrite=False):
    if fields is None:
        fields = source_df.columns.to_list()

    clean = source_df[fields].where(source_df[fields].notna(), None)
    info_map = clean.to_dict("index")

    updated = missing = unchanged = 0

    for sym in tqdm(lib.list_symbols(), desc="Adding metadata"):
        info = info_map.get(sym)
        if info is None:
            missing += 1
            continue

        existing = lib.read_metadata(sym).metadata or {}
        new_meta = dict (existing)
        for k, v in info.items():
            if overwrite or k not in new_meta:
                new_meta[k] = v

        if new_meta == existing:
            unchanged += 1
            continue

        lib.write_metadata(sym, new_meta)
        updated +=1

    print(f'Updated {updated} symbols, {unchanged} unchanged symbols, {missing} missing symbols')
    return {"updated": updated, "unchanged": unchanged, "missing": missing}

In [274]:
stocks_meta = (stocks.loc[stocks["table"] == "SEP",
                                ["ticker", "permaticker", "sector", "firstpricedate" , "lastpricedate"]]
                                .drop_duplicates("ticker")
                                .set_index("ticker"))
print(stocks_meta.head())
add_metadata(stocks_lib, stocks_meta, fields=["sector", "firstpricedate", "lastpricedate"])

        permaticker       sector firstpricedate lastpricedate
ticker                                                       
FRVO        6399080         None     2026-05-13    2026-05-15
BWIV.U      6399053         None     2026-05-11    2026-05-14
WSE         6399045   Technology     2026-05-11    2026-05-15
SAGU.U      6399044  Industrials     2026-05-07    2026-05-14
HAWK        6399043  Industrials     2026-05-07    2026-05-15


Adding metadata: 100%|██████████| 503/503 [00:05<00:00, 91.39it/s] 

Updated 503 symbols, 0 unchanged symbols, 0 missing symbols


{'updated': 503, 'unchanged': 0, 'missing': 0}

In [273]:
removed_stocks_meta = (stocks.loc[stocks["table"] == "SEP",
                                ["ticker", "permaticker", "sector", "firstpricedate" , "lastpricedate"]]
                                .drop_duplicates("ticker")
                                .set_index("ticker"))
print(removed_stocks_meta.head())
add_metadata(removed_lib, removed_stocks_meta, fields=["sector", "firstpricedate", "lastpricedate"])

        permaticker       sector firstpricedate lastpricedate
ticker                                                       
FRVO        6399080         None     2026-05-13    2026-05-15
BWIV.U      6399053         None     2026-05-11    2026-05-14
WSE         6399045   Technology     2026-05-11    2026-05-15
SAGU.U      6399044  Industrials     2026-05-07    2026-05-14
HAWK        6399043  Industrials     2026-05-07    2026-05-15


Adding metadata: 100%|██████████| 596/596 [00:05<00:00, 103.50it/s]

Updated 596 symbols, 0 unchanged symbols, 0 missing symbols


{'updated': 596, 'unchanged': 0, 'missing': 0}

In [233]:
sym = "AAPL"
print(f"{sym} in lib?    ", sym in stocks_lib.list_symbols())
print(f"{sym} in source? ", sym in stocks_meta.index)

AAPL in lib?     True
AAPL in source?  True


In [235]:
fields = ["sector"]
clean = stocks_meta[fields].where(stocks_meta[fields].notna(), None)
info_map = clean.to_dict("index")

print("info_map size:", len(info_map))
print("AAPL in info_map:", "AAPL" in info_map)
print("info_map['AAPL']:", info_map.get("AAPL"))

# Now mimic the loop's first iteration
sample = next(iter(stocks_lib.list_symbols()))
print(f"\nfirst lib symbol: {sample!r}")
print(f"  type: {type(sample)}")
print(f"  in info_map: {sample in info_map}")
print(f"  info_map.get: {info_map.get(sample)}")

info_map size: 21775
AAPL in info_map: True
info_map['AAPL']: {'sector': 'Technology'}

first lib symbol: 'AIZ'
  type: <class 'str'>
  in info_map: True
  info_map.get: {'sector': 'Financial Services'}


In [200]:
stocks.loc[stocks["table"] == "SEP", "sector"].value_counts(dropna=False)

sector
Industrials               4176
Technology                3669
Healthcare                3157
Financial Services        3075
Consumer Cyclical         1978
Communication Services    1209
Basic Materials           1076
Real Estate               1075
Energy                     960
Consumer Defensive         827
Utilities                  426
None                       147
Name: count, dtype: int64

In [202]:
sectors = (
    stocks.loc[stocks["table"] == "SEP", ["ticker", "sector", "industry"]]
        .drop_duplicates("ticker")
        .set_index("ticker")
)
sectors

,sector,industry
ticker,,
FRVO,None,None
BWIV.U,None,None
WSE,Technology,Information Technology Services
SAGU.U,Industrials,Shell Companies
HAWK,Industrials,Aerospace & Defense
...,...,...
BBUCQ,Consumer Cyclical,Restaurants
NWGN1,Technology,Software - Application
CYRS1,Communication Services,Telecom Services


In [207]:
current_counts = sectors.loc[sectors.index.intersection(current_members), "sector"].value_counts()
removed_counts = sectors.loc[sectors.index.intersection(removed), "sector"].value_counts()

In [208]:
current_counts

sector
Technology                85
Industrials               75
Financial Services        70
Healthcare                58
Consumer Cyclical         47
Consumer Defensive        35
Utilities                 32
Real Estate               31
Basic Materials           26
Communication Services    22
Energy                    22
Name: count, dtype: int64

In [209]:
removed_counts

sector
Technology                97
Consumer Cyclical         84
Financial Services        76
Healthcare                61
Industrials               55
Energy                    51
Consumer Defensive        45
Communication Services    44
Basic Materials           42
Utilities                 29
Real Estate               12
Name: count, dtype: int64

In [210]:
comparison = pd.concat([current_counts.rename("current"),
                          removed_counts.rename("removed")], axis=1).fillna(0).astype(int)
comparison["total"] = comparison["current"] + comparison["removed"]
print(comparison.sort_values("total", ascending=False))

                        current  removed  total
sector                                         
Technology                   85       97    182
Financial Services           70       76    146
Consumer Cyclical            47       84    131
Industrials                  75       55    130
Healthcare                   58       61    119
Consumer Defensive           35       45     80
Energy                       22       51     73
Basic Materials              26       42     68
Communication Services       22       44     66
Utilities                    32       29     61
Real Estate                  31       12     43


In [239]:
for tickers in removed:
    if tickers in stocks_lib.list_symbols():
        stocks_lib.delete(tickers)
        print(f'{tickers} removed. {len(stocks_lib.list_symbols())} number of existing symbols')

CTRA removed. 1082 number of existing symbols
HOLX removed. 1081 number of existing symbols
PAYC removed. 1080 number of existing symbols
MTCH removed. 1079 number of existing symbols
MOH removed. 1078 number of existing symbols
LW removed. 1077 number of existing symbols
DAY removed. 1076 number of existing symbols
SOLS removed. 1075 number of existing symbols
MHK removed. 1074 number of existing symbols
LKQ removed. 1073 number of existing symbols
K removed. 1072 number of existing symbols
IPG removed. 1071 number of existing symbols
EMN removed. 1070 number of existing symbols
KMX removed. 1069 number of existing symbols
MKTX removed. 1068 number of existing symbols
ENPH removed. 1067 number of existing symbols
CZR removed. 1066 number of existing symbols
WBA removed. 1065 number of existing symbols
HES removed. 1064 number of existing symbols
ANSS removed. 1063 number of existing symbols
JNPR removed. 1062 number of existing symbols
DFS removed. 1061 number of existing symbols
TFX 